In [24]:
from tinygrad.shape.view import View, merge_dims, unravel, strides_for_shape
from tinygrad.helpers import prod, all_int, argsort, flatten, ceildiv

In [50]:
# first assume we have a view with a single shape

def check_mergability(v1: View, v2: View) -> bool:
  for v2shape, v2stride in zip(v2.shape, v2.strides):
    if not check_mergability_single(v1, v2shape, v2stride):
      return False
  return True

def check_mergability_single(v1: View, v2shape: int, v2stride) -> bool:
  # use st for strides, s for shapes
  terms = unravel(v1.shape, v2stride)
  print(f"{terms=}")


  # the shape, term and stride of the currently "merged" dimensions
  current_extent = []
  all_extents = []
  merged_shape = []
  merged_term = 0   # this is the merged term
  merged_stride = 1 # this is the stride in the current dimension

  # Calculate extents based on overflow
  for d1, (shape, stride, term) in reversed(list(enumerate(zip(v1.shape, v1.strides, terms)))):
    print("--------------")
    current_extent.insert(0, d1) # todo: turn into deque
    print(f"{shape=} {stride=} {term=} {current_extent=}")
    merged_shape.insert(0, shape)
    merged_term += term * prod(merged_shape[1:])
    print(f"{merged_shape=} {merged_term=} {merged_stride=}")
    # check if overflow is going to happen or not
    merged_size = prod(merged_shape)
    if merged_term * v2shape > merged_size: # overflow will happen, so merge in the next dimension
      print("overflow")
      merged_stride *= shape # this is the stride for the next dimension
    else:
      print("no overflow")
      # finish this merged block, and start a new one
      merged_shape = []
      all_extents.append(current_extent)
      current_extent = []

  # check the linear equations in each extent
  for extent in all_extents:
    extent_terms = [terms[j] for j in extent]
    extent_shape = [v1.shape[j] for j in extent]
    extent_canonical_strides = strides_for_shape(tuple(extent_shape))
    extent_actual_strides = [v1.strides[j] for j in extent]
    print(f"{extent=} {extent_terms=} {extent_shape=} {extent_canonical_strides=}")
    
    if len(extent) == 1:
      continue
    if len(extent) == 2:
      print("Length 2 extent; is condition satisfied?", end=' ')
      if v1.shape[extent[1]] * v1.strides[extent[1]] == v1.strides[extent[0]]:
        print("yes")
      else:
        print("no")
        return False
    else:
      # now we need to track the conditions more carefully
      last_terms = [0 for _ in extent]
      canonical_stride = sum([term * stride for term, stride in zip(extent_terms, extent_canonical_strides)])
      print(f"     {extent_shape=}")
      overflow_sets = set()
      for i in range(1, v2shape):
        current_terms = unravel(v1.shape, i * canonical_stride)
        overflow = tuple(j for j in range(len(extent)) if current_terms[j] < last_terms[j])
        if overflow:
          overflow_sets.add(overflow)
        print(f"{i=} {current_terms=} {overflow=}")

        last_terms = current_terms

      print("Check conditions on overflow sets")
      for overflow_set in overflow_sets:
        print(f"Checking conditions for {overflow_set=}")
        overflow_coefficients = [0 for _ in extent_shape]
        for i in overflow_set:
          shape = extent_shape[i]
          overflow_coefficients[i] -= shape
          overflow_coefficients[i-1] += 1
        overflow_coefficients = tuple(overflow_coefficients)
        print(f"{overflow_coefficients=}")
        condition = sum([stride * coefficient for stride, coefficient in zip(extent_actual_strides, overflow_coefficients)])
        if condition != 0:
          print("Condition not satisfied")
          return False
        else:
          print("Condition satisfied")

    return True


# check_mergability(View.create((2, 5, 3), (15, 3, 1)), View.create((6, ), (3, )))
print("More difficult case")
check_mergability(View.create((10, 5, 3), (15, 3, 1)), View.create((6, ), (4, )))
print("My counterexample")
check_mergability(View.create((10, 9, 4), (8 * 11 + 4* 13, 11, 13)), View.create((6, ), (9, )))


# TODO: this doesn't work, because no extent is created, because the last thing overflows
# check_mergability(View.create((2, 5, 3), (15, 3, 1)), View.create((6, ), (4, )))


More difficult case
terms=[0, 1, 1]
--------------
shape=3 stride=1 term=1 current_extent=[2]
merged_shape=[3] merged_term=1 merged_stride=1
overflow
--------------
shape=5 stride=3 term=1 current_extent=[1, 2]
merged_shape=[5, 3] merged_term=4 merged_stride=3
overflow
--------------
shape=10 stride=15 term=0 current_extent=[0, 1, 2]
merged_shape=[10, 5, 3] merged_term=4 merged_stride=15
no overflow
extent=[0, 1, 2] extent_terms=[0, 1, 1] extent_shape=[10, 5, 3] extent_canonical_strides=(15, 3, 1)
     extent_shape=[10, 5, 3]
i=1 current_terms=[0, 1, 1] overflow=()
i=2 current_terms=[0, 2, 2] overflow=()
i=3 current_terms=[0, 4, 0] overflow=(2,)
i=4 current_terms=[1, 0, 1] overflow=(1,)
i=5 current_terms=[1, 1, 2] overflow=()
Check conditions on overflow sets
Checking conditions for overflow_set=(1,)
overflow_coefficients=(1, -5, 0)
Condition satisfied
Checking conditions for overflow_set=(2,)
overflow_coefficients=(0, 1, -3)
Condition satisfied
My counterexample
terms=[0, 2, 1]
------

True